# Agentic RAG with LangGraph

This notebook demonstrates building an Agentic RAG system using LangGraph with autonomous agents.

## Setup

In [ ]:
!pip install langgraph langchain langchain-openai langchain-community
!pip install grandalf

## Import Libraries

In [ ]:
import os
from langgraph.graph import StateGraph, END
from langchain_community.document_loaders import TextLoader
from langchain_community.tools import Tool
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from pydantic import BaseModel
from typing import List, TypedDict, Optional
import json


## Define Agent State

In [2]:
class AgentState(TypedDict):
    """State for the Agentic RAG agent."""
    question: str
    plan: List[str]
    retrieved_docs: List
    evaluation: str
    final_answer: Optional[str]
    iterations: int

## Load Document

In [3]:
# Create sample document
sample_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that enhances 
Large Language Models by retrieving relevant information from external 
knowledge bases. RAG addresses key limitations of LLMs including 
hallucinations, knowledge cutoff, and lack of source attribution.

The basic RAG pipeline consists of three components:
1. Retrieval: Finding relevant documents
2. Augmentation: Adding context to the prompt
3. Generation: Producing the final response

RAG is particularly useful for:
- Question answering systems
- Knowledge base chatbots
- Enterprise search
- Customer support automation
"""

# Save sample document
with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

# Load document
loader = TextLoader("sample_doc.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


## Create Tools

In [4]:
# Initialize components
llm = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Create vector store (using existing documents)
vectorstore = Chroma.from_documents(
    documents=documents,  # From previous notebook
    embedding=embeddings
)

def vector_search(query: str) -> str:
    """Search the knowledge base."""
    docs = vectorstore.similarity_search(query, k=4)
    return "\n\n".join([f"Doc {i+1}: {doc.page_content[:200]}" for i, doc in enumerate(docs)])

def analyze_context(query: str, context: str) -> str:
    """Analyze if context answers the question."""
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nDoes this answer the question?"
    return llm.invoke(prompt)

tools = [
    Tool(
        name="vector_search",
        func=vector_search,
        description="Search internal knowledge base"
    ),
    Tool(
        name="analyze_context",
        func=analyze_context,
        description="Analyze if retrieved context answers the question"
    )
]

## Define Agent Nodes

In [5]:
def planner_node(state: AgentState) -> AgentState:
    """Plan retrieval strategy."""
    prompt = f"""Given this question: {state['question']}\n\nCreate a plan to answer it.\nRespond as a JSON list of steps."""
    response = llm.invoke(prompt)
    try:
        plan = json.loads(response)
    except:
        plan = ["Search knowledge base", "Analyze results", "Generate answer"]
    return {"plan": plan, "iterations": state.get("iterations", 0)}

def retriever_node(state: AgentState) -> AgentState:
    """Execute retrieval."""
    docs = vectorstore.similarity_search(state['question'], k=4)
    return {"retrieved_docs": docs}

def evaluator_node(state: AgentState) -> AgentState:
    """Evaluate retrieval quality."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Question: {state['question']}\n\nContext:\n{context}\n\nIs this sufficient?\n\nRespond: 'sufficient', 'need_more', or 'refine'"""
    evaluation = llm.invoke(prompt).content.strip().lower()
    return {"evaluation": evaluation}

def generator_node(state: AgentState) -> AgentState:
    """Generate final answer."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Based on the context, answer the question.\n\nContext:\n{context}\n\nQuestion: {state['question']}\n\nAnswer:"""
    answer = llm.invoke(prompt)
    return {"final_answer": answer}

## Build the Graph

In [6]:
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("retrieve", retriever_node)
workflow.add_node("evaluate", evaluator_node)
workflow.add_node("generate", generator_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "retrieve")
workflow.add_edge("retrieve", "evaluate")

def should_continue(state: AgentState) -> str:
    """Decide whether to continue or finish."""
    if state.get("evaluation", "") == "need_more":
        if state.get("iterations", 0) < 2:
            return "retry"
    return "generate"

workflow.add_conditional_edges(
    "evaluate",
    should_continue,
    {"retry": "planner", "generate": "generate"}
)

workflow.add_edge("generate", END)

app = workflow.compile()

## Run Agentic RAG

In [7]:
result = app.invoke({
    "question": "What are the key benefits of RAG over pure LLMs?",
    "plan": [],
    "retrieved_docs": [],
    "evaluation": "",
    "final_answer": None,
    "iterations": 0
})

print("Question:", result["question"])
print("\nPlan:", result.get("plan", []))
print("\nIterations:", result.get("iterations", 0))
print("\nAnswer:", result["final_answer"])

Question: What are the key benefits of RAG over pure LLMs?

Plan: ['Search knowledge base', 'Analyze results', 'Generate answer']

Iterations: 0

Answer: content="The key benefits of RAG over pure LLMs include:\n\n1. Addressing hallucinations: RAG reduces the likelihood of hallucinations by retrieving relevant information from external knowledge bases, making the model's responses more grounded in reality.\n2. Overcoming knowledge cutoff: RAG can access a vast amount of knowledge beyond the model's training data, reducing the knowledge cutoff issue that can lead to outdated or incomplete information.\n3. Providing source attribution: RAG allows for source attribution, enabling users to understand the origin of the information provided, which is not possible with pure LLMs." additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-05T15:50:57.033571Z', 'done': True, 'done_reason': 'stop', 'total_duration': 841784542, 'load_duration': 53434542, 'prompt_eval_cou

## Visualize the Graph

In [8]:
# Display the graph structure
print(app.get_graph().draw_ascii())

           +-----------+       
           | __start__ |       
           +-----------+       
                  *            
                  *            
                  *            
            +---------+        
            | planner |        
            +---------+        
           ***        ...      
          *              .     
        **                ...  
+----------+                 . 
| retrieve |              ...  
+----------+             .     
           ***        ...      
              *      .         
               **  ..          
            +----------+       
            | evaluate |       
            +----------+       
                  .            
                  .            
                  .            
            +----------+       
            | generate |       
            +----------+       
                  *            
                  *            
                  *            
            +---------+        
        

## Try Different Queries

In [9]:
queries = [
    "How does RAG reduce hallucinations?",
    "What is the difference between classic RAG and agentic RAG?"
]

for q in queries:
    result = app.invoke({
        "question": q,
        "plan": [],
        "retrieved_docs": [],
        "evaluation": "",
        "final_answer": None,
        "iterations": 0
    })
    print(f"Q: {q}")
    print(f"A: {result['final_answer']}\n")

Q: How does RAG reduce hallucinations?
A: content='RAG reduces hallucinations by retrieving relevant information from external knowledge bases, which helps to ground the model\'s responses in actual facts and data, rather than relying on its own internal knowledge or generating entirely new information. This process prevents the model from "hallucinating" or making things up, as it is forced to rely on verifiable sources to inform its responses.' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-05T15:51:08.580868Z', 'done': True, 'done_reason': 'stop', 'total_duration': 576212750, 'load_duration': 53219459, 'prompt_eval_count': 169, 'prompt_eval_duration': 82521000, 'eval_count': 74, 'eval_duration': 430801708, 'logprobs': None, 'model_name': 'llama3.2'} id='run--019cbeb2-15e3-7e81-b735-4c50c6bec6fd-0' usage_metadata={'input_tokens': 169, 'output_tokens': 74, 'total_tokens': 243}

Q: What is the difference between classic RAG and agentic RAG?
A: conte

## Next Steps

- Add more tools (web search, calculator)
- Implement multi-agent collaboration
- Add memory for conversation history
- Add error handling and retry logic